# AgriVision : DINOv3 ViT-S/16 + tête de segmentation sur Agriculture-Vision 2021

**Prérequis (une seule fois)**
1. Avoir l'accès au modèle https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m (licence DINOv3 acceptée).
2. Avoir un secret Colab `HF_TOKEN` (icône 🔑), avec l'accès notebook activé.
3. Runtime → Modifier le type d'exécution → GPU **L4** ou **A100**.

**Run 2 (« recette complète »)**
- toutes les données : 56 944 tuiles d'entraînement, évaluation sur les 18 334 tuiles du val officiel, comme dans les papiers ;
- les 4 derniers blocs de DINOv3 sont ajustés ;
- les classes rares sont tirées plus souvent et pèsent plus dans la loss ;
- mosaïques et zooms aléatoires pendant l'entraînement ;
- à la fin, prédiction moyennée sur 8 rotations et symétries.

La première fois, les données complètes (~9 Go) sont préparées puis sauvegardées sur Drive. Si la session coupe, relancer les cellules : l'entraînement reprend depuis le dernier checkpoint.

In [ ]:
# --- Configuration ---
DRIVE_DIR = "/content/drive/MyDrive/agrivision"     # données + runs (persistant)
CODE_SOURCE = "git"                                  # "drive" : copie depuis CODE_DIR ; "git" : clone REPO_URL
CODE_DIR = f"{DRIVE_DIR}/code"                       # dossier du projet (contient agrivision/)
REPO_URL = "https://github.com/JulesV19/AgriVision.git"

DATA_MODE = "full"            # "full" : tout le dataset ; "subset" : 12k/3k/2k tuiles comme le run 1
SAVE_DATA_TO_DRIVE = True     # ~9 Go en mode full ; False = reconstruire depuis S3 à chaque session (~20-30 min)

RUN_NAME = "vits16_ft4_full"
EPOCHS, BATCH_SIZE = 12, 32
LR, BACKBONE_LR, UNFREEZE_LAST = 1e-3, 3e-5, 4
BACKBONE = "facebook/dinov3-vits16-pretrain-lvd1689m"
REF_RUN = "vits16_frozen_512"  # run 1, pour comparer les courbes

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os

def sh(cmd):
    # Comme `!cmd`, mais lève une erreur si la commande échoue (sinon « Exécuter tout » continue).
    ip = get_ipython()
    ip.system(f"set -o pipefail; {cmd}")
    if ip.user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"commande en échec (code {ip.user_ns['_exit_code']}), voir la sortie ci-dessus : {cmd}")

os.makedirs(DRIVE_DIR, exist_ok=True)
if CODE_SOURCE == "git":
    sh(f"rm -rf /content/AgriVision && git clone -q {REPO_URL} /content/AgriVision")
else:
    sh(f'rm -rf /content/AgriVision && cp -r "{CODE_DIR}" /content/AgriVision')
%cd /content/AgriVision
sh("nvidia-smi --query-gpu=name,memory.total --format=csv && nproc")
WORKERS = os.cpu_count()

In [ ]:
sh("pip install -q -r requirements.txt")

In [ ]:
# Connexion HF + vérification de l'accès au modèle gated
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
from transformers import AutoConfig
print(AutoConfig.from_pretrained(BACKBONE).model_type, "OK")

## Données
Si une archive valide existe sur Drive, on la recopie simplement. Sinon, on télécharge l'archive depuis S3 (21 Go, environ 2 min), on l'extrait sans le test officiel ni le NIR, puis on la convertit au format compact.

In [ ]:
FULL = DATA_MODE == "full"
DATA = "/content/agv_full" if FULL else "/content/agv_subset"
DATA_TAR = f"{DRIVE_DIR}/{os.path.basename(DATA)}.tar"
EVAL_SPLIT = "val" if FULL else "test"
S3 = "https://intelinair-data-releases.s3.amazonaws.com/agriculture-vision/cvpr_challenge_2021/supervised/Agriculture-Vision-2021.tar.gz"
RAW = "/content/raw/Agriculture-Vision-2021"

def data_ok():
    return os.path.exists(f"{DATA}/meta.json")

if not data_ok() and os.path.exists(DATA_TAR):
    print(f"{DATA_TAR} : {os.path.getsize(DATA_TAR) / 1e9:.2f} Go")
    sh(f'tar xf "{DATA_TAR}" -C /content || true')
    if not data_ok():
        print("-> archive invalide ou incomplète : suppression et reconstruction")
        os.remove(DATA_TAR)

if not data_ok():
    if not os.path.exists("/content/raw/.done"):
        sh("rm -rf /content/raw && mkdir -p /content/raw && df -h /content | tail -1")
        sh("apt-get -qq install -y aria2 pigz > /dev/null")
        sh(f"aria2c -x 16 -s 16 -k 64M -c --file-allocation=none --show-console-readout=false "
           f"--summary-interval=15 --console-log-level=warn -d /content -o agv2021.tar.gz {S3}")
        sh("tar -I pigz -xf /content/agv2021.tar.gz -C /content/raw --exclude='*/test/*' --exclude='*/images/nir/*' "
           "--checkpoint=50000 --checkpoint-action=dot")
        sh("rm /content/agv2021.tar.gz && touch /content/raw/.done")
    mode = "--full" if FULL else "--n-train 12000 --n-val 3000 --n-test 2000"
    sh(f"rm -rf {DATA} && python -m agrivision.prepare_subset --src {RAW} --out {DATA} {mode} --workers {WORKERS}")
    sh("rm -rf /content/raw")
    if SAVE_DATA_TO_DRIVE:
        # écriture atomique sur Drive : jamais d'archive partielle en cas de coupure
        name = os.path.basename(DATA)
        sh(f'tar cf /content/{name}.tar -C /content {name} && cp /content/{name}.tar "{DATA_TAR}.tmp" '
           f'&& mv "{DATA_TAR}.tmp" "{DATA_TAR}" && rm /content/{name}.tar')
sh(f"du -sh {DATA}/* && python -c \"import json; m=json.load(open('{DATA}/meta.json')); "
   f"print({{k: v['n_tiles'] for k, v in m['splits'].items()}})\"")

## Entraînement
La première epoch affiche sa durée, ainsi que la part du temps passée à attendre les données (si elle est élevée, c'est le CPU qui limite). Le mIoU affiché à chaque epoch est calculé sur 4 000 tuiles du val, pour aller vite. Le score final est calculé plus bas sur tout le val.

In [ ]:
RUN = f"{DRIVE_DIR}/runs/{RUN_NAME}"
sh(f'python -m agrivision.train --data {DATA} --out "{RUN}" --backbone {BACKBONE} '
   f'--unfreeze-last {UNFREEZE_LAST} --epochs {EPOCHS} --batch-size {BATCH_SIZE} '
   f'--lr {LR} --backbone-lr {BACKBONE_LR} --workers {WORKERS}')

In [ ]:
# Évaluation finale sur tout le split (18 334 tuiles en mode full), sans puis avec TTA x8 (~8x plus long)
for tta in (1, 8):
    sh(f'python -m agrivision.train --data {DATA} --out "{RUN}" --eval-only --split {EVAL_SPLIT} '
       f'--tta {tta} --batch-size {BATCH_SIZE} --workers {WORKERS}')

## (Optionnel) Référence CNN : DeepLabV3+ EfficientNet-b3 (~13 M paramètres)
Même données et même recette. C'est l'équivalent du modèle à 0,513 de la 2ᵉ place 2024, en RGB seul. À lancer dans une autre session si le temps manque.

In [ ]:
RUN_CNN = False
if RUN_CNN:
    RUN_REF_CNN = f"{DRIVE_DIR}/runs/deeplabv3p_effb3_full"
    sh(f'python -m agrivision.train --data {DATA} --out "{RUN_REF_CNN}" --arch smp --smp-arch DeepLabV3Plus '
       f'--smp-encoder efficientnet-b3 --epochs {EPOCHS} --batch-size 16 --lr 3e-4 --backbone-lr 1e-4 --workers {WORKERS}')
    for tta in (1, 8):
        sh(f'python -m agrivision.train --data {DATA} --out "{RUN_REF_CNN}" --eval-only --split {EVAL_SPLIT} '
           f'--tta {tta} --batch-size 16 --workers {WORKERS}')

In [ ]:
import json, matplotlib.pyplot as plt
def load_log(run):
    path = f"{DRIVE_DIR}/runs/{run}/log.jsonl"
    return [json.loads(l) for l in open(path)] if os.path.exists(path) else []
log, ref = load_log(RUN_NAME), load_log(REF_RUN)
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot([e["loss"] for e in log]); ax[0].set_title("loss train"); ax[0].set_xlabel("epoch")
for c in log[0]["val_iou"]:
    ax[1].plot([e["val_iou"][c] for e in log], label=c)
ax[1].set_title("IoU val par classe"); ax[1].legend(fontsize=7, ncol=2)
ax[2].plot([e["val_miou"] for e in log], "k-", lw=2, label=RUN_NAME)
if ref:
    ax[2].plot([e["val_miou"] for e in ref], "k--", label=f"{REF_RUN} (sous-ensemble)")
ax[2].set_title("mIoU val"); ax[2].legend(); plt.show()
for f in sorted(os.listdir(RUN)):
    if f.startswith("eval_"):
        r = json.load(open(f"{RUN}/{f}"))
        print(f"{f}: mIoU {r['miou']:.4f} sur {r['n_tiles']} tuiles | "
              + " ".join(f"{c[:6]}={v:.2f}" for c, v in r["iou"].items()))

In [ ]:
# Visualisation : image | vérité terrain | prédiction
import argparse, numpy as np, torch
from agrivision import CLASSES, MEAN, STD
from agrivision.data import AgriVisionSubset, decode_labels
from agrivision.train import build_model

device = "cuda"
ckpt = torch.load(f"{RUN}/best.pt", map_location="cpu")
model = build_model(argparse.Namespace(**ckpt["model_args"]), len(CLASSES)).to(device).eval()
model.load_state_dict(ckpt["model"], strict=False)
ds = AgriVisionSubset(DATA, EVAL_SPLIT)

PALETTE = np.array([[0,0,0],[230,25,75],[255,225,25],[60,180,75],[245,130,48],
                    [145,30,180],[0,130,200],[70,240,240],[240,50,230]], np.uint8)

def show(idxs):
    fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 4 * len(idxs)))
    for row, i in zip(axes, idxs):
        img, lbl = ds[i]
        target, valid = decode_labels(lbl[None].to(device), len(CLASSES))
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            pred = model(img[None].to(device)).argmax(1)[0].cpu().numpy()
        gt = (target[0].float() * torch.arange(len(CLASSES), device=device)[:, None, None]).amax(0).cpu().numpy().astype(int)
        rgb = (img.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)
        inv = ~valid[0].cpu().numpy()
        gt_rgb, pred_rgb = PALETTE[gt], PALETTE[pred]
        gt_rgb[inv] = pred_rgb[inv] = 128
        for a, im, t in zip(row, [rgb, gt_rgb, pred_rgb], [ds.ids[i], "vérité", "prédiction"]):
            a.imshow(im); a.set_title(t, fontsize=8); a.axis("off")
    plt.tight_layout(); plt.show()
    print("  ".join(f"{c}={tuple(int(x) for x in PALETTE[k])}" for k, c in enumerate(CLASSES)))

show(np.random.choice(len(ds), 4, replace=False))